# BRCA1 ESM-DMS Experimental Pipeline

This notebook runs the class-based BRCA1 workflow from `data/mavedb_data`: MaveDB counts and functional scores are keyed by `hgvs_nt`, mutated protein sequences are reconstructed from the BRCA1 wildtype protein reference where the nucleotide mutation has an unambiguous amino-acid consequence, embeddings are pooled and cached, DeltaSAE features are inferred, and model fitness is compared with MaveDB functional scores.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from esmDMS import CellularDMSInput, ESMDMSConfig, esmDMS

DATA_DIR = REPO_ROOT / "data" / "mavedb_data"
ANALYSIS_DIR = REPO_ROOT / "data" / "esm_data_analysis" / "BRCA1_experimental"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

for directory in (SEQUENCE_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT

## Configure BRCA1

`SAE_EMBEDDING_TYPE` selects which saved pooled embedding cache trains or feeds the sparse autoencoder. Use `"mean_pool"` or `"max_pool"`. `EMBEDDING_MODEL` can be an ESM-2 Hugging Face model such as `"facebook/esm2_t6_8M_UR50D"` or an ESMC model identifier supported by the local environment.

In [ ]:
BRCA1_INPUT = CellularDMSInput(
    reference_nuc_path=DATA_DIR / "BRCA1_reference_sequence.dat",
    mavedb_csv_path=DATA_DIR / "BRCA1_counts.csv",
    scores_csv_path=DATA_DIR / "BRCA1_scores.csv",
    reference_kind="protein",
    primary_key="hgvs_nt",
)

EMBEDDING_MODEL = "facebook/esm2_t6_8M_UR50D"
REPRESENTATIVE_LAYER = 6
SAE_EMBEDDING_TYPE = "mean_pool"  # "mean_pool" or "max_pool"
ABSTRACTION_METHOD = "DeltaSAE"
NORM_SCHEME = "none"

SAE_PARAMS = {
    "n_features": 256,
    "sparsity_coeff": 1e-3,
    "sparsity_mode": "topk",  # "normal", "topk", or "batchtopk"
    "k": 16,
    "epochs": 200,
    "batch_size": 64,
    "train_frac": 0.8,
    "lr": 1e-3,
    "seed": 42,
    "run_label": f"{SAE_EMBEDDING_TYPE}_topk16",
    "norm_scheme": NORM_SCHEME,
    # "pretrained_model_path": SEQUENCE_DIR / "sae_models" / "existing_model.pt",
}

config = ESMDMSConfig(
    embedding_model=EMBEDDING_MODEL,
    embedding_type=SAE_EMBEDDING_TYPE,
    local_or_disk="both",
    save_dir=str(SEQUENCE_DIR),
    dataset_name="BRCA1",
)

runner = esmDMS(input_data=BRCA1_INPUT, config=config)
runner

## Process Counts And Scores

The parser keeps `hgvs_nt` as `SequenceIndex`, stores a separate key-to-protein-sequence map, loads functional scores keyed by `hgvs_nt`, and skips rows that cannot produce a unique protein sequence from a protein-only reference, such as intronic HGVS entries or ambiguous codon effects.

In [ ]:
runner.process_raw_data(drop_stop_codons=True)

processing_summary = pd.DataFrame([{
    "dataset": "BRCA1",
    "reference_kind": runner.reference_kind,
    "count_rows": len(runner.sequence_dataframe),
    "mutation_keys_in_counts": runner.sequence_dataframe["SequenceIndex"].nunique(),
    "protein_sequence_keys": len(runner.sequence_to_protein_sequence),
    "replicates": runner.sequence_dataframe["Replicate"].nunique(),
    "generations": sorted(runner.sequence_dataframe["Generation"].unique()),
    "score_rows": len(runner.scores_dataframe),
    "skipped_counts": runner.sequence_metadata.attrs.get("skipped_counts", {}),
}])
processing_summary.to_csv(TABLE_DIR / "BRCA1_processing_summary.csv", index=False)
processing_summary

In [ ]:
runner.sequence_dataframe.head()

In [ ]:
runner.scores_dataframe.head()

## Embed Mutated And Wildtype Proteins

Embedding writes `mean_pool`, `max_pool`, and `per_residue` caches for each layer. The wildtype protein is embedded even though it does not appear as a count row, because DeltaSAE subtracts its SAE representation.

In [ ]:
RUN_LOCAL_EMBEDDINGS = False
CREATE_EMBEDDING_JOB = False
SUBMIT_JOBS = False

if RUN_LOCAL_EMBEDDINGS:
    runner.embed_all_sequences(layer=REPRESENTATIVE_LAYER)

if CREATE_EMBEDDING_JOB:
    embedding_job = runner.create_embedding_batch_job(
        job_dir=SEQUENCE_DIR / "embedding_batch_jobs",
        n_chunks=40,
        max_active_jobs=4,
        job_name="brca1_esm_embed",
        partition="any_cpu",
        mem="24G",
        time="08:00:00",
        python_executable="python3",
        scratch_root="/scr",
        submit=SUBMIT_JOBS,
    )
    display(pd.DataFrame([{
        "script_path": str(embedding_job["script_path"]),
        "payload_path": str(embedding_job["payload_path"]),
        "job_id": embedding_job["job_id"],
    }]))

In [ ]:
MERGE_EMBEDDING_OUTPUTS = False

if MERGE_EMBEDDING_OUTPUTS:
    runner.merge_embedding_batch_outputs(
        job_dir=SEQUENCE_DIR / "embedding_batch_jobs",
        layer="all",
        save_layers=True,
    )

cache_status = pd.DataFrame([
    {
        "layer": REPRESENTATIVE_LAYER,
        "embedding_type": embedding_type,
        "path": str(runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type)),
        "exists": runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type).exists(),
    }
    for embedding_type in ("mean_pool", "max_pool", "per_residue")
])
cache_status.to_csv(TABLE_DIR / "BRCA1_embedding_cache_status.csv", index=False)
cache_status

## Train Or Load DeltaSAE Features

`create_feature_space(..., method="DeltaSAE")` trains the selected SAE variant on the pooled embeddings, keeps SAE dimensions with activation frequency strictly between 0 and 1, and returns `SAE(mutant) - SAE(wildtype)` for every `hgvs_nt` mutation key.

In [ ]:
RUN_DELTA_SAE = False

if RUN_DELTA_SAE:
    delta_sae_features = runner.create_feature_space(
        layer=REPRESENTATIVE_LAYER,
        method=ABSTRACTION_METHOD,
        method_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
    )
    print(f"DeltaSAE feature count: {len(delta_sae_features)}")
    print(f"DeltaSAE dimensions: {len(next(iter(delta_sae_features.values())))}")

In [ ]:
PLOT_SAE_QA = False

if PLOT_SAE_QA:
    runner.visualize_sae_reconstructions(
        layer=REPRESENTATIVE_LAYER,
        method_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_reconstruction_QA.png",
    )
    plt.show()

## Infer Selection On DeltaSAE Features

The inferred model fitness is `1 + sel_coeffs_joint dot DeltaSAE`.

In [ ]:
RUN_INFERENCE = False
LOAD_COMPLETED_INFERENCE = False
inference_result = None

if RUN_INFERENCE:
    inference_result = runner.run_feature_inference(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        abstraction_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
    )
elif LOAD_COMPLETED_INFERENCE:
    inference_result = runner.load_inference_results(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        embedding_type=SAE_EMBEDDING_TYPE,
    )

if inference_result is not None:
    inference_summary = pd.DataFrame([{
        "layer": REPRESENTATIVE_LAYER,
        "embedding_type": SAE_EMBEDDING_TYPE,
        "abstraction_method": ABSTRACTION_METHOD,
        "n_replicates": inference_result.s.shape[0],
        "n_dimensions": inference_result.s.shape[1],
        "gamma_opt": inference_result.gamma_opt,
        "s_joint_mean": inference_result.s_joint.mean(),
        "s_joint_std": inference_result.s_joint.std(),
    }])
    inference_summary.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_inference_summary.csv", index=False)
    display(inference_summary)
else:
    print("Set RUN_INFERENCE or LOAD_COMPLETED_INFERENCE to True after DeltaSAE features are available.")

## Regularization Diagnostics

In [ ]:
PLOT_REGULARIZATION = False

if PLOT_REGULARIZATION:
    fig, regularization_df = runner.plot_regularization_curve(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        abstraction_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_regularization.png",
    )
    regularization_df.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_regularization.csv", index=False)
    plt.show()

## Compare Inferred Fitness With MaveDB Scores

In [ ]:
PLOT_SCORE_COMPARISON = False

if PLOT_SCORE_COMPARISON:
    fig, score_comparison_df, score_stats = runner.plot_functional_score_comparison(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        abstraction_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        score_col="score",
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_fitness_vs_score.png",
    )
    score_comparison_df.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_fitness_vs_scores.csv", index=False)
    display(pd.DataFrame([score_stats]))
    plt.show()

## Cross-Replicate Consistency

In [ ]:
PLOT_REPLICATE_CONSISTENCY = False

if PLOT_REPLICATE_CONSISTENCY:
    runner.plot_rep_sel_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        label="BRCA1 DeltaSAE",
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_selection_replicate_scatter.png",
    )
    plt.show()

    runner.plot_rep_fit_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        label="BRCA1 DeltaSAE",
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_fitness_replicate_scatter.png",
    )
    plt.show()

## Selection Coefficient Distribution

In [ ]:
PLOT_SELECTION_COEFFICIENTS = False

if PLOT_SELECTION_COEFFICIENTS:
    fig, coef_df = runner.plot_selection_coefficient_distribution(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_selection_coefficients.png",
    )
    coef_df.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_selection_coefficients.csv", index=False)
    plt.show()